# 02 · Modélisation
**Membre 3 — Axes 4 et 5 · Pipeline : variables → split chronologique → baselines → modèles → validation croisée temporelle → test (une seule fois)**

Entrée : `data/processed/features_conso.csv` (préparé par le membre 2). Cible : `Consommation_kWh` du jour (kWh). Chaque ligne contient uniquement des informations connues à la fin du jour précédent (retards J-1 et J-7, calendrier).

Règles respectées : split d'abord, préparation dans un `Pipeline`, baselines à battre, au moins 3 modèles, CV temporelle à 5 plis avec écart-type, réglage d'un hyperparamètre (`max_depth`), **test utilisé une seule fois à la fin**.

**Métriques (axe 5)** : métrique principale = **MAE** (erreur moyenne en kWh, lisible par un décideur, peu sensible aux jours atypiques). Garde-fou = **RMSE** (elle amplifie les grosses erreurs, utile pour repérer les jours très mal prévus). Le R² et le MAPE ne sont pas utilisés : le R² seul ne dit rien en kWh, et le MAPE est instable si la consommation s'approche de 0.

## 0 · Imports et configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error

RANDOM_STATE = 42

DATA_PROCESSED = Path("data/processed")
FIGURES = Path("reports/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (12, 4)

Le fichier `features_conso.csv` est préparé dans le notebook 01. Sur Colab, la cellule suivante demande de l'importer.

In [ ]:
# Fichier produit par 01_cadrage_eda (membre 2). Sur Google Colab, on l'importe s'il est absent.
fichier_csv = DATA_PROCESSED / "features_conso.csv"

if not fichier_csv.exists():
    try:
        from google.colab import files
        DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
        uploaded = files.upload()   # choisir features_conso.csv sur l'ordinateur
        nom_csv = next(nom for nom in uploaded if nom.endswith(".csv"))
        Path(nom_csv).replace(fichier_csv)
    except ImportError:
        raise FileNotFoundError("Lancer d'abord 01_cadrage_eda.ipynb : il crée data/processed/features_conso.csv")

## 1 · Chargement et variables
Le fichier du membre 2 contient déjà `conso_J_minus_1` et `conso_J_minus_7`, construits **par date** (vérifié : chaque retard correspond bien à la consommation du jour J-1 / J-7 quand ce jour existe). On renomme les colonnes et on ajoute `Moyenne_7_jours` (moyenne des 7 jours précédents), car c'est une baseline forte à tester.

In [ ]:
# Fichier préparé par le membre 2 : une ligne = un jour complet,
# avec la consommation du jour (cible) et les retards J-1 et J-7 déjà construits PAR DATE.
conso = pd.read_csv(DATA_PROCESSED / "features_conso.csv", parse_dates=["Date"])
conso = conso.sort_values("Date").reset_index(drop=True)

# On garde les noms de colonnes utilisés dans la suite du notebook
conso = conso.rename(columns={
    "conso_J_minus_1": "Consommation_veille",
    "conso_J_minus_7": "Consommation_semaine_precedente",
    "est_we": "week_end",
})


def ajouter_moyenne_7_jours(conso):
    # Moyenne des 7 jours PRÉCÉDENTS : le shift(1) évite d'inclure le jour à prédire.
    # On travaille sur un calendrier complet : un jour absent devient NaN, pas un faux retard.
    serie = conso.set_index("Date")["Consommation_kWh"]
    calendrier = pd.date_range(serie.index.min(), serie.index.max(), freq="D")
    serie = serie.reindex(calendrier)

    # min_periods=4 : on accepte une fenêtre avec au plus 3 jours manquants
    moyenne = serie.shift(1).rolling(window=7, min_periods=4).mean()

    conso = conso.copy()
    conso["Moyenne_7_jours"] = moyenne.reindex(conso["Date"]).values
    return conso.dropna().reset_index(drop=True)


donnees = ajouter_moyenne_7_jours(conso)

print("Lignes dans le fichier du membre 2 :", len(conso))
print("Journées utilisables (avec Moyenne_7_jours) :", len(donnees))
display(donnees.head())
display(donnees.isna().sum())

**Interprétation** : on passe de 1 205 lignes à 1 163 journées utilisables car la fenêtre de 7 jours exige au moins 4 jours connus avant la date prédite. Aucune valeur manquante ensuite. Toutes les variables sont connues à la fin du jour J-1 : pas de fuite.

## 2 · Split chronologique (le test = la période la plus récente)
80 % du passé pour apprendre, 20 % du futur pour le test final. **Pas de mélange aléatoire** : le futur ne doit jamais servir à prédire le passé.

In [ ]:
donnees = donnees.sort_values("Date").reset_index(drop=True)

limite = int(len(donnees) * 0.8)
train = donnees.iloc[:limite].copy()
test = donnees.iloc[limite:].copy()

colonnes_numeriques = [
    "Consommation_veille",
    "Consommation_semaine_precedente",
    "Moyenne_7_jours",
    "week_end",
]
colonnes_categorielles = ["jour_semaine", "mois"]
variables = colonnes_numeriques + colonnes_categorielles

X_train = train[variables]
y_train = train["Consommation_kWh"]
X_test = test[variables]
y_test = test["Consommation_kWh"]

print("Train :", len(train), "jours,", train["Date"].min().date(), "->", train["Date"].max().date())
print("Test  :", len(test), "jours,", test["Date"].min().date(), "->", test["Date"].max().date())

## 3 · Modèles et pipelines
- **Baseline 1** : demain = aujourd'hui (*situation actuelle* du cadrage).
- **Baseline 2** : demain = même jour la semaine dernière.
- **Baseline 3** : demain = moyenne des 7 derniers jours.
- **Régression linéaire** : `jour_semaine` et `mois` sont des catégories (`OneHotEncoder`), les variables numériques sont standardisées, le tout dans le `Pipeline` donc ajusté sur le train uniquement.
- **Arbre de décision** et **Random Forest** : `max_depth` réglé par validation croisée temporelle.

In [ ]:
# Régression linéaire avec préparation dans le pipeline
preparation_lineaire = ColumnTransformer([
    ("num", StandardScaler(), colonnes_numeriques),
    ("cat", OneHotEncoder(handle_unknown="ignore"), colonnes_categorielles),
])

modele_lineaire = Pipeline([
    ("prep", preparation_lineaire),
    ("reg", LinearRegression()),
])

# Validation croisée temporelle : le passé prédit toujours le futur
tscv = TimeSeriesSplit(n_splits=5)

# Réglage de max_depth de l'arbre (sur le train uniquement, MAE)
grille_arbre = GridSearchCV(
    DecisionTreeRegressor(random_state=RANDOM_STATE),
    param_grid={"max_depth": [2, 3, 4, 5, 8]},
    cv=tscv,
    scoring="neg_mean_absolute_error",
)
grille_arbre.fit(X_train, y_train)

# Réglage de max_depth de la forêt
grille_foret = GridSearchCV(
    RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
    param_grid={"max_depth": [3, 5, 8]},
    cv=tscv,
    scoring="neg_mean_absolute_error",
)
grille_foret.fit(X_train, y_train)

print("Arbre - meilleure profondeur :", grille_arbre.best_params_)
print("Forêt - meilleure profondeur :", grille_foret.best_params_)

for nom, grille in [("Arbre", grille_arbre), ("Forêt", grille_foret)]:
    resume = pd.DataFrame(grille.cv_results_)[["param_max_depth", "mean_test_score", "std_test_score"]].copy()
    resume["MAE_val_moy"] = -resume["mean_test_score"]
    resume["MAE_val_ecart_type"] = resume["std_test_score"]
    print("\n" + nom + " : MAE de validation selon max_depth")
    display(resume[["param_max_depth", "MAE_val_moy", "MAE_val_ecart_type"]].round(3))

**Interprétation** : l'arbre retient `max_depth=4` et la forêt `max_depth=3`. Pour l'arbre, une profondeur de 8 dégrade la validation (MAE 6,60 contre 5,68) : c'est du surapprentissage. Pour la forêt, la MAE varie peu (5,20 à 5,29) : le réglage compte moins. Les écarts-types (0,4 à 0,9 kWh) sont du même ordre que les écarts entre profondeurs, donc ces différences ne sont pas très fiables.

## 4 · Comparaison par validation croisée temporelle (sur le train uniquement)

In [ ]:
# Une chaîne de caractères = baseline (on recopie la colonne). Sinon = vrai modèle.
modeles = {
    "Baseline : veille": "Consommation_veille",
    "Baseline : même jour semaine précédente": "Consommation_semaine_precedente",
    "Baseline : moyenne 7 jours": "Moyenne_7_jours",
    "Régression linéaire": modele_lineaire,
    "Arbre de décision": grille_arbre.best_estimator_,
    "Random Forest": grille_foret.best_estimator_,
}

lignes = []

for nom, modele in modeles.items():
    maes_train = []
    maes_val = []

    for idx_train, idx_val in tscv.split(X_train):
        X_tr = X_train.iloc[idx_train]
        X_va = X_train.iloc[idx_val]
        y_tr = y_train.iloc[idx_train]
        y_va = y_train.iloc[idx_val]

        if isinstance(modele, str):
            pred_tr = X_tr[modele]
            pred_va = X_va[modele]
            type_modele = "Baseline"
        else:
            modele.fit(X_tr, y_tr)
            pred_tr = modele.predict(X_tr)
            pred_va = modele.predict(X_va)
            type_modele = "Modèle"

        maes_train.append(mean_absolute_error(y_tr, pred_tr))
        maes_val.append(mean_absolute_error(y_va, pred_va))

    lignes.append({
        "Modèle": nom,
        "Type": type_modele,
        "MAE_train_moy": np.mean(maes_train),
        "MAE_val_moy": np.mean(maes_val),
        "MAE_val_ecart_type": np.std(maes_val),
    })

tableau_cv = pd.DataFrame(lignes)
display(tableau_cv.sort_values("MAE_val_moy").round(3))

**Interprétation** :
- La meilleure baseline est la **moyenne 7 jours** (MAE de validation 5,06 kWh). La baseline « veille » (situation actuelle) fait 6,02 kWh.
- Le Random Forest (5,20 ± 0,43) est le meilleur vrai modèle, mais il ne bat pas la moyenne 7 jours (5,06 ± 0,55) : l'écart est plus petit que l'écart-type, donc la différence n'est pas fiable.
- Face à la baseline « veille », le Random Forest gagne environ 14 % en validation (5,20 contre 6,02).
- La régression linéaire est instable (écart-type 2,0 kWh). L'arbre montre un écart train/validation (4,92 contre 5,68) : léger surapprentissage.
- La MAE de validation peut différer de celle du test : les premiers plis s'entraînent sur moins d'un an de données et ne voient pas toutes les saisons.

## 5 · Choix du modèle (d'après la CV, pas d'après le test) et évaluation finale
Le modèle retenu est celui qui a la plus faible `MAE_val_moy` parmi les **vrais modèles**. Le test est utilisé ici, **une seule fois**.

In [ ]:
tableau_modeles = tableau_cv[tableau_cv["Type"] == "Modèle"].sort_values("MAE_val_moy")
tableau_baselines = tableau_cv[tableau_cv["Type"] == "Baseline"].sort_values("MAE_val_moy")

nom_retenu = tableau_modeles.iloc[0]["Modèle"]
nom_baseline = tableau_baselines.iloc[0]["Modèle"]       # meilleure baseline selon la CV
colonne_baseline = modeles[nom_baseline]
colonne_veille = modeles["Baseline : veille"]             # situation actuelle du cadrage

print("Modèle retenu (CV) :", nom_retenu)
print("Meilleure baseline (CV) :", nom_baseline)

# Entraînement sur tout le train, puis UN SEUL passage sur le test
modele_final = modeles[nom_retenu]
modele_final.fit(X_train, y_train)

pred_modele = modele_final.predict(X_test)
pred_baseline = X_test[colonne_baseline]
pred_veille = X_test[colonne_veille]

mae_modele = mean_absolute_error(y_test, pred_modele)
rmse_modele = np.sqrt(mean_squared_error(y_test, pred_modele))
mae_baseline = mean_absolute_error(y_test, pred_baseline)
rmse_baseline = np.sqrt(mean_squared_error(y_test, pred_baseline))
mae_veille = mean_absolute_error(y_test, pred_veille)
rmse_veille = np.sqrt(mean_squared_error(y_test, pred_veille))

gain_vs_baseline = (mae_baseline - mae_modele) / mae_baseline * 100
gain_vs_veille = (mae_veille - mae_modele) / mae_veille * 100

print("\n=== RÉSULTATS SUR LE TEST (jamais vu) ===")
print("MAE veille (situation actuelle)  :", round(mae_veille, 2), "kWh   | RMSE :", round(rmse_veille, 2))
print("MAE meilleure baseline           :", round(mae_baseline, 2), "kWh   | RMSE :", round(rmse_baseline, 2))
print("MAE modèle                       :", round(mae_modele, 2), "kWh   | RMSE :", round(rmse_modele, 2))
print("Amélioration de la MAE vs veille (critère : -20 %)  :", round(gain_vs_veille, 1), "%")
print("Amélioration de la MAE vs meilleure baseline        :", round(gain_vs_baseline, 1), "%")
print("MAE modèle / consommation moyenne du test :",
      round(mae_modele / y_test.mean() * 100, 1), "%")

**Interprétation** : sur des données jamais vues (3 février → 25 novembre 2010), notre prévision du lendemain se trompe en moyenne de **4,19 kWh**, soit **17,3 %** de la consommation moyenne. La baseline « veille » (situation actuelle) se trompe de 4,82 kWh : le modèle l'améliore de **13,1 %**. En revanche, la moyenne 7 jours fait 4,18 kWh : le modèle ne la bat pas (−0,1 %).

**Le critère de succès provisoire (−20 % de MAE) n'est donc pas atteint**, ni contre la veille (13,1 %), ni contre la meilleure baseline. C'est un résultat recevable, à présenter honnêtement : avec seulement des retards et du calendrier sur un seul foyer, le modèle apporte peu par rapport à une simple moyenne glissante.

## 6 · Diagnostic : erreurs et importance des variables

In [ ]:
resultats = test[["Date", "Consommation_kWh", "jour_semaine", "mois"]].copy()
resultats = resultats.rename(columns={"Consommation_kWh": "reel"})
resultats["pred_modele"] = pred_modele
resultats["pred_baseline"] = pred_baseline.values
resultats["pred_veille"] = pred_veille.values
resultats["erreur_abs"] = (resultats["reel"] - resultats["pred_modele"]).abs()

plt.figure()
plt.plot(resultats["Date"], resultats["reel"], label="Réel", linewidth=1.2)
plt.plot(resultats["Date"], resultats["pred_modele"], label="Modèle : " + nom_retenu, linewidth=1.0)
plt.plot(resultats["Date"], resultats["pred_baseline"], label="Meilleure baseline", linewidth=0.8, alpha=0.6)
plt.title("Consommation réelle vs prévue sur la période de test")
plt.xlabel("Date")
plt.ylabel("Consommation (kWh/jour)")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES / "05_reel_vs_prevu.png", dpi=150, bbox_inches="tight")
plt.show()

**Interprétation** : le modèle suit le niveau général mais lisse les pics et les creux. Il ne voit pas les ruptures brutales (absence du foyer, retour de voyage).

In [ ]:
# Où le modèle se trompe-t-il le plus ?
noms_jours = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]
erreur_par_jour = resultats.groupby("jour_semaine")["erreur_abs"].mean()

plt.figure(figsize=(8, 4))
plt.bar(range(7), erreur_par_jour.reindex(range(7)).values)
plt.xticks(range(7), noms_jours)
plt.title("Erreur moyenne du modèle selon le jour de la semaine")
plt.xlabel("Jour")
plt.ylabel("Erreur absolue moyenne (kWh)")
plt.tight_layout()
plt.savefig(FIGURES / "06_erreur_par_jour.png", dpi=150, bbox_inches="tight")
plt.show()

print("10 plus grosses erreurs :")
display(resultats.nlargest(10, "erreur_abs")[["Date", "reel", "pred_modele", "erreur_abs"]].round({"reel": 2, "pred_modele": 2, "erreur_abs": 2}))

**Interprétation** : les plus grosses erreurs (13 à 22 kWh) tombent sur des jours de rupture : fin février / début mars 2010 (jours à 12-13 kWh, qui ressemblent à une absence du foyer, à vérifier), et plusieurs jours isolés très hauts (18/10, 04/11) ou très bas (21/11, 16/10) que rien dans les retards ne permet d'anticiper. Ce que le modèle ne voit pas : la météo, la présence au foyer, les jours fériés et les vacances.

In [ ]:
# Importance des variables
if nom_retenu == "Régression linéaire":
    noms = modele_final.named_steps["prep"].get_feature_names_out()
    importances = pd.Series(modele_final.named_steps["reg"].coef_, index=noms)
    titre = "Coefficients de la régression linéaire (variables standardisées)"
else:
    importances = pd.Series(modele_final.feature_importances_, index=variables)
    titre = "Importance des variables (" + nom_retenu + ")"

ordre = importances.abs().sort_values().index
importances = importances.reindex(ordre)

plt.figure(figsize=(8, 6))
plt.barh(importances.index, importances.values)
plt.title(titre)
plt.tight_layout()
plt.savefig(FIGURES / "07_importance_variables.png", dpi=150, bbox_inches="tight")
plt.show()
print(importances.sort_values(ascending=False).round(2))

**Interprétation** : `Moyenne_7_jours` domine nettement (importance 0,61), puis `Consommation_veille` (0,25). `Consommation_semaine_precedente`, `week_end`, `jour_semaine` et `mois` pèsent peu. C'est cohérent avec l'analyse exploratoire (forte corrélation avec J-1 et J-7) et avec le fait que la meilleure baseline est elle-même une moyenne des jours précédents.

## 7 · Sauvegarde des résultats pour le notebook 03 (décision et ROI)
On sauvegarde la baseline « veille » (situation actuelle) en plus de la meilleure baseline, pour que le membre 1 puisse calculer le ROI par rapport à la situation actuelle.

In [ ]:
resultats[["Date", "reel", "pred_modele", "pred_baseline", "pred_veille"]].to_csv(
    DATA_PROCESSED / "resultats_test.csv", index=False
)

metriques = pd.DataFrame({
    "indicateur": ["modele_retenu", "baseline_reference", "mae_modele", "mae_baseline", "mae_veille",
                   "rmse_modele", "rmse_baseline", "rmse_veille", "conso_moyenne_test"],
    "valeur": [nom_retenu, nom_baseline, mae_modele, mae_baseline, mae_veille,
               rmse_modele, rmse_baseline, rmse_veille, y_test.mean()],
})
metriques.to_csv(DATA_PROCESSED / "metriques_test.csv", index=False)
tableau_cv.to_csv(DATA_PROCESSED / "comparaison_cv.csv", index=False)
print("Résultats sauvegardés dans", DATA_PROCESSED)